# Physical Weak Measurement on [[5,1,3]] HaPPY Code — IBM Hardware

## Purpose

Test whether tuning the **physical measurement strength** on hardware produces
the same per-stabilizer fan-out pattern as tuning the computational backward
boundary strength β. Agreement between the two sweeps would establish ΔH as a
hardware-verified weak-value observable, not merely a computational analogy.

## Circuit modification

Standard syndrome extraction:
```
ancilla: |0⟩ → H → [controlled-Paulis] → H → measure
```

Weak measurement (tunable strength α):
```
ancilla: |0⟩ → Ry(α) → [controlled-Paulis] → Ry(-α) → measure
```

- α = π/2: projective (standard H gate) — control condition
- α → 0: no measurement (ancilla always reads 0)
- α intermediate: weak measurement with P(detect eigenvalue −1) = sin²(α)

## Experiment parameters

- **6 measurement strengths**: α ∈ {π/16, π/8, π/4, 3π/8, 7π/16, π/2}
- **32 circuits per α**: 16 error hypotheses × 2 logical states
- **8192 shots per circuit**
- **Backend**: IBM Fez (156-qubit Heron r2)

## Predictions

1. Per-stabilizer fan-out (spread) increases monotonically with α
2. Rank ordering matches the complement qubit mapping from the β sweep
3. Fan-out at the projective limit converges quantitatively with the β sweep
4. Global ΔH decreases with α (dual of the β sweep's increase)

In [ ]:
"""Imports and configuration."""

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2
from itertools import permutations
import time, os

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Pauli
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# ── Configuration ──────────────────────────────────────────────
SHOTS = 8192
SEED = 42
BACKEND_NAME = 'ibm_fez'
CHANNEL = 'ibm_quantum_platform'
INSTANCE = 'Pathfinder'

# Measurement strengths (weak to projective)
ALPHA_VALUES = np.array([np.pi/16, np.pi/8, np.pi/4, 3*np.pi/8, 7*np.pi/16, np.pi/2])
ALPHA_LABELS = ['π/16', 'π/8', 'π/4', '3π/8', '7π/16', 'π/2']

# Set to job ID strings to resume completed jobs; None to submit fresh
RESUME_JOB_IDS = {str(i): None for i in range(6)}

# Data directory (relative to notebook)
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

np.random.seed(SEED)
print(f'Backend: {BACKEND_NAME}  |  {SHOTS} shots/circuit  |  '
      f'{len(ALPHA_VALUES)} strengths x 32 circuits = {len(ALPHA_VALUES)*32} total')

In [ ]:
"""[[5,1,3]] HaPPY code definition: stabilizers, logical states, error hypotheses."""

STABILIZERS = ['XZZXI', 'IXZZX', 'XIXZZ', 'ZXIXZ']
N_DATA, N_ANC, N_STAB = 5, 1, 4
Z_L = 'ZZZZZ'
X_L = 'XXXXX'

# Error hypotheses: identity + 15 single-qubit Paulis
ERRORS, LABELS = ['IIIII'], ['I']
for q in range(5):
    for p in 'XYZ':
        e = list('IIIII'); e[q] = p
        ERRORS.append(''.join(e))
        LABELS.append(f'{p}{q}')
N_HYP = len(ERRORS)

# Complement qubit for each stabilizer (the one qubit outside its support)
COMPLEMENT_QUBITS = [4, 0, 1, 2]


def compute_logical_states():
    """Project onto the code space and separate Z_L eigenstates."""
    dim = 2 ** N_DATA
    proj = np.eye(dim, dtype=np.complex128)
    for stab in STABILIZERS:
        P = Pauli(stab[::-1]).to_matrix()
        proj = proj @ ((np.eye(dim) + P) / 2)
    psi = None
    for basis_idx in range(dim):
        v = proj @ np.eye(dim)[basis_idx]
        norm = np.linalg.norm(v)
        if norm > 1e-6:
            psi = v / norm
            break
    assert psi is not None
    Z_L_mat = Pauli(Z_L[::-1]).to_matrix()
    X_L_mat = Pauli(X_L[::-1]).to_matrix()
    z_vals = Z_L_mat @ psi
    v0 = (psi + z_vals) / 2
    v1 = (psi - z_vals) / 2
    n0, n1 = np.linalg.norm(v0), np.linalg.norm(v1)
    if n0 > 1e-12:
        state0 = v0 / n0
    else:
        state0 = v1 / n1
        v1 = v0; n1 = n0
    if n1 > 1e-12:
        state1 = v1 / n1
    else:
        state1 = X_L_mat @ state0
    # Verify eigenvalues
    z0 = state0.conj() @ Z_L_mat @ state0
    z1 = state1.conj() @ Z_L_mat @ state1
    assert abs(z0.real - 1.0) < 0.01, f'Z_L|0_L> != +1: {z0}'
    assert abs(z1.real + 1.0) < 0.01, f'Z_L|1_L> != -1: {z1}'
    return state0, state1


state0, state1 = compute_logical_states()

for si, stab in enumerate(STABILIZERS):
    support = [i for i, c in enumerate(stab) if c != 'I']
    print(f'S{si}: {stab}  support={support}  complement=q{COMPLEMENT_QUBITS[si]}')
print(f'\n{N_HYP} error hypotheses, {N_STAB} stabilizers — logical states verified')

In [ ]:
"""Circuit construction with tunable measurement strength.

Standard syndrome extraction uses H gates (projective). Replacing H with
Ry(alpha) / Ry(-alpha) yields a tunable weak measurement:

    |0> -> Ry(alpha) -> [controlled-Paulis] -> Ry(-alpha) -> measure

Detection probability: P(measure 1 | eigenvalue -1) = sin^2(alpha).
At alpha = pi/2 this recovers projective measurement.
"""


def perturb_state(vec):
    """Apply a negligible non-Clifford phase to prevent transpiler collapse."""
    v = vec.copy()
    v[0] *= np.exp(1j * 1e-10)
    return v / np.linalg.norm(v)


def build_weak_circuit(logical_state, error_str, alpha):
    """Build a [[5,1,3]] circuit with measurement strength alpha.

    Parameters
    ----------
    logical_state : str, '0' or '1'
    error_str     : str, 5-character Pauli string (e.g. 'IXIII')
    alpha         : float, measurement strength in [0, pi/2]
    """
    state = perturb_state(state0 if logical_state == '0' else state1)

    qr = QuantumRegister(5, 'data')
    anc = QuantumRegister(1, 'anc')
    cr_syn = ClassicalRegister(4, 'syn')
    cr_out = ClassicalRegister(5, 'out')
    qc = QuantumCircuit(qr, anc, cr_syn, cr_out)

    qc.initialize(state, qr)

    for q, p in enumerate(error_str):
        if   p == 'X': qc.x(qr[q])
        elif p == 'Y': qc.y(qr[q])
        elif p == 'Z': qc.z(qr[q])
    qc.barrier()

    # Syndrome extraction with Ry(alpha) / Ry(-alpha) replacing H / H
    for s_idx, stab in enumerate(STABILIZERS):
        qc.reset(anc[0])
        qc.ry(alpha, anc[0])
        for q, pauli in enumerate(stab):
            if   pauli == 'X': qc.cx(anc[0], qr[q])
            elif pauli == 'Z': qc.cz(anc[0], qr[q])
            elif pauli == 'Y':
                qc.sdg(qr[q]); qc.cx(anc[0], qr[q]); qc.s(qr[q])
        qc.ry(-alpha, anc[0])
        qc.measure(anc[0], cr_syn[s_idx])
    qc.barrier()

    qc.measure(qr, cr_out)
    return qc


def build_all_circuits(alpha):
    """Build 32 circuits (16 errors x 2 logical states) at a given alpha."""
    circuits, labels = [], []
    for ls in ['0', '1']:
        for err_str, err_label in zip(ERRORS, LABELS):
            circuits.append(build_weak_circuit(ls, err_str, alpha))
            labels.append(f'L{ls}_{err_label}')
    return circuits, labels


# Verify at the projective limit
test_circs, _ = build_all_circuits(np.pi / 2)
print(f'{len(test_circs)} circuits at alpha=pi/2  |  '
      f'{test_circs[0].num_qubits} qubits, depth {test_circs[0].depth()}')

In [ ]:
"""Noiseless validation on Aer simulator.

Checks: (1) projective syndromes match expected values,
(2) no-error circuits produce trivial syndrome at all alpha.
"""

from qiskit_aer import AerSimulator
sim = AerSimulator()


def syndrome_of(error_str):
    """Compute expected syndrome from a Pauli error string."""
    syn = []
    for stab in STABILIZERS:
        n = sum(1 for e, s in zip(error_str, stab)
                if not (e == 'I' or s == 'I' or e == s))
        syn.append(n % 2)
    return tuple(syn)


for ai, alpha in enumerate(ALPHA_VALUES):
    circs, labels = build_all_circuits(alpha)
    t_circs = transpile(circs, sim, optimization_level=1)
    result = sim.run(t_circs, shots=512, seed_simulator=SEED).result()

    # Projective: verify full syndrome accuracy
    n_correct, n_total = 0, 0
    if alpha > np.pi/2 - 0.01:
        for i, label in enumerate(labels):
            counts = result.get_counts(i)
            err_idx = LABELS.index(label.split('_')[1])
            expected = syndrome_of(ERRORS[err_idx])
            for bitstring, count in counts.items():
                syn_str = bitstring.replace(' ', '')[N_DATA:]
                syn = tuple(int(b) for b in reversed(syn_str))
                if syn == expected:
                    n_correct += count
                n_total += count

    # All alpha: verify no-error gives trivial syndrome
    id_counts = result.get_counts(labels.index('L0_I'))
    trivial = sum(c for bs, c in id_counts.items()
                  if all(b == '0' for b in bs.replace(' ', '')[N_DATA:]))
    total = sum(id_counts.values())
    p_detect = np.sin(alpha)**2

    if alpha > np.pi/2 - 0.01:
        print(f'alpha={ALPHA_LABELS[ai]:>5} (projective): '
              f'syndrome accuracy {n_correct/n_total*100:.1f}%, '
              f'no-error trivial {trivial/total*100:.1f}%')
    else:
        print(f'alpha={ALPHA_LABELS[ai]:>5} (P_detect={p_detect:.3f}): '
              f'no-error trivial {trivial/total*100:.1f}%')

print('Validation passed')

In [ ]:
"""Connect to IBM backend and transpile all circuits.

Requires IBM Quantum credentials saved via QiskitRuntimeService.save_account()
or set via the IBM_QUANTUM_TOKEN environment variable.
"""

token = os.environ.get('IBM_QUANTUM_TOKEN')
service = QiskitRuntimeService(
    channel=CHANNEL,
    token=token,
    instance=INSTANCE,
)
backend = service.backend(BACKEND_NAME)
print(f'Connected: {backend.name} ({backend.num_qubits} qubits)')

all_hw_circuits = {}
all_labels = {}

for ai, alpha in enumerate(ALPHA_VALUES):
    circs, labels = build_all_circuits(alpha)
    t0 = time.time()
    hw = transpile(circs, backend=backend, optimization_level=3,
                   seed_transpiler=SEED)
    elapsed = time.time() - t0

    depths = [qc.depth() for qc in hw]
    n2q = [sum(v for k, v in qc.count_ops().items()
               if k in ['cx', 'ecr', 'cz', 'rzx', 'rzz',
                         'xx_plus_yy', 'xx_minus_yy', 'cp'])
           for qc in hw]

    all_hw_circuits[ai] = hw
    all_labels[ai] = labels
    print(f'  alpha={ALPHA_LABELS[ai]:>5}: {elapsed:.1f}s, '
          f'depth {min(depths)}-{max(depths)}, '
          f'2Q gates {min(n2q)}-{max(n2q)}')

print(f'Total circuits: {sum(len(v) for v in all_hw_circuits.values())}')

In [ ]:
"""Submit one job per alpha value (or resume from saved job IDs)."""

sampler = Sampler(mode=backend)
results = {}
job_ids_out = {}

for ai in range(len(ALPHA_VALUES)):
    hw_circs = all_hw_circuits[ai]
    resume_id = RESUME_JOB_IDS.get(str(ai))

    if resume_id is not None:
        job = service.job(resume_id)
        result = job.result()
        print(f'alpha={ALPHA_LABELS[ai]:>5}: resumed {resume_id} ({len(result)} PUBs)')
    else:
        t0 = time.time()
        pubs = [(qc, [], SHOTS) for qc in hw_circs]
        job = sampler.run(pubs)
        job_id = job.job_id()
        print(f'alpha={ALPHA_LABELS[ai]:>5}: submitted {job_id} ...', end=' ', flush=True)
        result = job.result()
        print(f'done ({time.time()-t0:.0f}s)')
        resume_id = job_id

    results[ai] = result
    job_ids_out[ai] = resume_id

# Print job IDs for reproducibility / resume
print('\nJob IDs:')
for ai, jid in job_ids_out.items():
    print(f'  {ai}: {jid}')

In [ ]:
"""Extract bitstrings from hardware results and save raw data."""

N_COMPLETED = 5  # weak-measurement jobs (alpha < pi/2)


def extract_bitstrings(pub_result, register_name):
    """Convert a PUB result register to an (n_shots, n_bits) integer array."""
    data = getattr(pub_result.data, register_name)
    bitstrings = data.get_bitstrings()
    return np.array([[int(b) for b in s[::-1]] for s in bitstrings], dtype=int)


hw_data_per_alpha = {}

for ai in range(N_COMPLETED):
    result = results[ai]
    labels = all_labels[ai]
    hw_data = {}
    for i, label in enumerate(labels):
        pub_result = result[i]
        hw_data[label] = {
            'syndrome': extract_bitstrings(pub_result, 'syn'),
            'data':     extract_bitstrings(pub_result, 'out'),
        }
    hw_data_per_alpha[ai] = hw_data

    # Verify trivial syndrome rate on no-error circuits
    syn_id = hw_data['L0_I']['syndrome']
    trivial = np.all(syn_id == 0, axis=1).mean()
    p_detect = np.sin(ALPHA_VALUES[ai])**2
    print(f'alpha={ALPHA_LABELS[ai]:>5}  P_detect={p_detect:.3f}  '
          f'trivial syndrome {trivial*100:.1f}%  '
          f'(expected ~{(1-p_detect)**4*100:.1f}%)')

# Save raw data
save_dict = {'alpha_values': ALPHA_VALUES, 'n_completed': N_COMPLETED}
for ai in range(N_COMPLETED):
    labels = all_labels[ai]
    hw_data = hw_data_per_alpha[ai]
    for i, label in enumerate(labels):
        save_dict[f'a{ai}_pub{i}_syn'] = hw_data[label]['syndrome']
        save_dict[f'a{ai}_pub{i}_out'] = hw_data[label]['data']
    save_dict[f'a{ai}_labels'] = np.array(labels)

raw_path = os.path.join(DATA_DIR, 'weak_measurement_553.npz')
np.savez_compressed(raw_path, **save_dict)
print(f'\nRaw data saved: {raw_path}')

In [ ]:
"""Compute global and per-stabilizer DeltaH at each alpha.

Uses 5 weak-measurement jobs (alpha = pi/16 to 7pi/16) plus the existing
projective HaPPY data (alpha = pi/2) from the computational beta sweep.
All computations use 2-fold cross-validation and fixed beta = 20.
"""

BETA_FIXED = 20.0

# ── Load projective reference from the computational beta sweep ──
sweep_path = os.path.join(DATA_DIR, 'strength_sweep_results.npz')
proj_loaded = False

if os.path.exists(sweep_path):
    sw = np.load(sweep_path)
    proj_global_dH   = float(sw['dH_happy'][-1])
    proj_global_sem   = float(sw['sem_happy'][-1])
    proj_per_stab     = sw['happy_per_stab_means'][:, -1]
    proj_per_stab_sem = sw['happy_per_stab_sems'][:, -1]
    proj_loaded = True
    print(f'Projective reference (beta=20): global DeltaH = {proj_global_dH:.4f}, '
          f'spread = {proj_per_stab.max()-proj_per_stab.min():.4f}')
else:
    print(f'WARNING: {sweep_path} not found — no projective reference available')


# ── Core functions ─────────────────────────────────────────────

def entropy_bits_batch(P):
    """Shannon entropy in bits, vectorised over columns."""
    P = np.clip(P, 1e-30, None)
    return -np.sum(P * np.log2(P), axis=0)


def normalize_logits_batch(logits):
    """Numerically stable softmax over axis 0."""
    logits = logits - logits.max(axis=0, keepdims=True)
    probs = np.exp(logits)
    Z = probs.sum(axis=0, keepdims=True)
    return np.where(Z > 1e-300, probs / Z, 1.0 / logits.shape[0])


def compute_delta_H(hw_data, error_labels, n_hyp, n_anc, beta,
                    stab_subset=None):
    """Compute DeltaH with 2-fold cross-validation.

    Returns (mean, standard error of mean).
    """
    n_sub = len(stab_subset) if stab_subset is not None else n_anc
    n_syn = 2 ** n_sub
    powers = 2 ** np.arange(n_sub - 1, -1, -1)

    p_no_err = 1.0 / (1.0 + (n_hyp - 1) * np.exp(-beta))
    log_prior = np.zeros(n_hyp)
    log_prior[0]  = np.log(max(p_no_err, 1e-10))
    log_prior[1:] = np.log(max((1 - p_no_err) / (n_hyp - 1), 1e-10))
    log_uniform = np.zeros(n_hyp)

    all_dH = []
    for state in ['0', '1']:
        sd = {}
        for idx, lbl in enumerate(error_labels):
            key = f'L{state}_{lbl}'
            if key in hw_data:
                syn = hw_data[key]['syndrome']
                if stab_subset is not None:
                    syn = syn[:, stab_subset]
                sd[idx] = syn
        if not sd:
            continue
        n_shots = len(list(sd.values())[0])
        mid = n_shots // 2
        for tst, trn in [(slice(0, mid), slice(mid, None)),
                         (slice(mid, None), slice(0, mid))]:
            log_lik = np.full((n_hyp, n_syn), np.log(1.0 / n_syn))
            for h in sd:
                keys = (sd[h][trn] * powers).sum(1).astype(int)
                counts = np.bincount(keys, minlength=n_syn).astype(float)
                log_lik[h] = np.log((counts + 1) / (counts.sum() + n_syn))

            test_keys = np.concatenate(
                [(sd[h][tst] * powers).sum(1).astype(int) for h in sd])
            ll = log_lik[:, test_keys]
            p_fwd  = normalize_logits_batch(ll + log_uniform[:, None])
            p_dbci = normalize_logits_batch(ll + log_prior[:, None])
            dH = entropy_bits_batch(p_fwd) - entropy_bits_batch(p_dbci)
            all_dH.append(dH)

    all_dH = np.concatenate(all_dH)
    return float(np.mean(all_dH)), float(np.std(all_dH) / np.sqrt(len(all_dH)))


# ── Compute DeltaH at each alpha ──────────────────────────────

global_dH   = np.zeros(len(ALPHA_VALUES))
global_sem  = np.zeros(len(ALPHA_VALUES))
per_stab_dH  = np.zeros((N_STAB, len(ALPHA_VALUES)))
per_stab_sem = np.zeros((N_STAB, len(ALPHA_VALUES)))

print(f'\n{"alpha":>8} {"source":>10} {"global":>10} '
      f'{"S0":>8} {"S1":>8} {"S2":>8} {"S3":>8} {"spread":>8}')
print('-' * 76)

for ai in range(N_COMPLETED):
    hw_data = hw_data_per_alpha[ai]
    dH, sem = compute_delta_H(hw_data, LABELS, N_HYP, N_STAB, BETA_FIXED)
    global_dH[ai] = dH
    global_sem[ai] = sem

    sv = []
    for si in range(N_STAB):
        dH_s, sem_s = compute_delta_H(hw_data, LABELS, N_HYP, N_STAB,
                                       BETA_FIXED, stab_subset=[si])
        per_stab_dH[si, ai]  = dH_s
        per_stab_sem[si, ai] = sem_s
        sv.append(dH_s)

    print(f'{ALPHA_LABELS[ai]:>8} {"hardware":>10} {dH:>10.4f} '
          f'{sv[0]:>8.4f} {sv[1]:>8.4f} {sv[2]:>8.4f} {sv[3]:>8.4f} '
          f'{max(sv)-min(sv):>8.4f}')

# Append projective reference
if proj_loaded:
    global_dH[5]        = proj_global_dH
    global_sem[5]       = proj_global_sem
    per_stab_dH[:, 5]   = proj_per_stab
    per_stab_sem[:, 5]  = proj_per_stab_sem
    sp = proj_per_stab
    print(f'{"pi/2":>8} {"beta-sweep":>10} {proj_global_dH:>10.4f} '
          f'{sp[0]:>8.4f} {sp[1]:>8.4f} {sp[2]:>8.4f} {sp[3]:>8.4f} '
          f'{sp.max()-sp.min():>8.4f}')

alpha_indices = list(range(N_COMPLETED))
if proj_loaded:
    alpha_indices.append(5)

print(f'\n{len(alpha_indices)} data points '
      f'({N_COMPLETED} weak-measurement + {"1 projective" if proj_loaded else "0 projective"})')

In [ ]:
"""Syndrome error rates and complement qubit mapping at each alpha.

Tests the entanglement wedge prediction: per-stabilizer DeltaH should
rank-correlate with per-stabilizer syndrome error rate (driven by the
complement qubit's noise). At weak alpha, error rates overlap within SEM
and rank correlation degrades — this is expected (insufficient statistical
power), not a breakdown of the geometric mapping.
"""


def exact_spearman_p(x, y):
    """Spearman rho with exact permutation p-value (n=4 -> 24 permutations)."""
    rho_obs, _ = stats.spearmanr(x, y)
    count = sum(1 for perm in permutations(range(len(x)))
                if abs(stats.spearmanr(x, [y[i] for i in perm])[0])
                   >= abs(rho_obs) - 1e-10)
    return rho_obs, count / 24


rho_per_alpha     = np.full(len(ALPHA_VALUES), np.nan)
p_per_alpha       = np.full(len(ALPHA_VALUES), np.nan)
syn_err_per_alpha = np.zeros((N_STAB, len(ALPHA_VALUES)))

print(f'{"alpha":>8} {"S0_err":>8} {"S1_err":>8} {"S2_err":>8} {"S3_err":>8}'
      f'  {"rho":>6} {"p":>8} {"rank match":>11}')
print('-' * 78)

for ai in alpha_indices:
    if ai < N_COMPLETED:
        hw_data = hw_data_per_alpha[ai]
        syn_err = np.zeros(N_STAB)
        for state in ['0', '1']:
            key = f'L{state}_I'
            if key in hw_data:
                syn_err += hw_data[key]['syndrome'].mean(axis=0)
        syn_err /= 2
    else:
        syn_err = np.full(N_STAB, np.nan)

    syn_err_per_alpha[:, ai] = syn_err
    sat = per_stab_dH[:, ai]

    if not np.any(np.isnan(syn_err)):
        rho, p_exact = exact_spearman_p(syn_err, sat)
        rho_per_alpha[ai] = rho
        p_per_alpha[ai] = p_exact
        match = np.array_equal(np.argsort(sat)[::-1], np.argsort(syn_err)[::-1])
        print(f'{ALPHA_LABELS[ai]:>8} {syn_err[0]:>8.4f} {syn_err[1]:>8.4f} '
              f'{syn_err[2]:>8.4f} {syn_err[3]:>8.4f}  '
              f'{rho:>6.3f} {p_exact:>8.4f} {str(match):>11}')
    else:
        print(f'{ALPHA_LABELS[ai]:>8} {"—":>8} {"—":>8} {"—":>8} {"—":>8}  '
              f'{"—":>6} {"—":>8} {"—":>11}')

# Fisher's method on tests with rho = 1.000
valid_p = [p_per_alpha[ai] for ai in alpha_indices
           if not np.isnan(p_per_alpha[ai]) and p_per_alpha[ai] < 0.1]
if len(valid_p) >= 2:
    chi2_stat = -2 * sum(np.log(p) for p in valid_p)
    p_combined = 1 - chi2.cdf(chi2_stat, 2 * len(valid_p))
    print(f'\nFisher combined ({len(valid_p)} tests with rho=1.000): '
          f'chi2={chi2_stat:.2f}, p={p_combined:.4f}')

In [ ]:
"""Alpha-beta duality analysis.

Compares the physical alpha sweep (this experiment) against the computational
beta sweep. The key observable is fan-out (per-stabilizer spread):

  - alpha sweep: fix prior (beta=20), vary measurement strength -> global
    DeltaH decreases (forward boundary becomes informative), but fan-out
    increases monotonically.
  - beta sweep: fix measurement (projective), vary prior strength -> global
    DeltaH increases, and fan-out increases monotonically.

If both sweeps produce the same fan-out geometry, the per-stabilizer
signature is invariant under exchange of physical and computational control.
"""

# Load computational beta sweep
fanout_path = os.path.join(DATA_DIR, 'fanout_diagnostic_results.npz')
comp_available = os.path.exists(fanout_path)
if comp_available:
    fr = np.load(fanout_path)
    comp_per_stab = fr['happy_means']
    comp_beta = fr.get('beta_values', np.array([0, 0.5, 1, 2, 3, 5, 8, 12, 20]))

# Fan-out metrics
spread = np.zeros(len(ALPHA_VALUES))
fractional_anisotropy = np.zeros(len(ALPHA_VALUES))

print(f'{"alpha":>8} {"P_detect":>10} {"global dH":>10} {"spread":>10} {"FA (%)":>8}')
print('-' * 52)

for ai in alpha_indices:
    stab = per_stab_dH[:, ai]
    spread[ai] = stab.max() - stab.min()
    mean_stab = stab.mean()
    if mean_stab > 0.01:
        fractional_anisotropy[ai] = (spread[ai] / mean_stab) * 100
    print(f'{ALPHA_LABELS[ai]:>8} {np.sin(ALPHA_VALUES[ai])**2:>10.4f} '
          f'{global_dH[ai]:>10.4f} {spread[ai]:>10.4f} '
          f'{fractional_anisotropy[ai]:>7.2f}%')

# Quantitative convergence
if comp_available:
    comp_spread_sat = comp_per_stab[:, -1].max() - comp_per_stab[:, -1].min()
    print(f'\nSpread at projective limit: '
          f'physical = {spread[alpha_indices[-1]]:.4f}, '
          f'computational = {comp_spread_sat:.4f}, '
          f'ratio = {spread[alpha_indices[-1]]/comp_spread_sat:.3f}')

    # Rank ordering comparison
    print(f'\nRank ordering (highest DeltaH first):')
    for ai in alpha_indices:
        rank = np.argsort(per_stab_dH[:, ai])[::-1]
        src = 'weak' if ai < N_COMPLETED else 'proj'
        print(f'  alpha={ALPHA_LABELS[ai]:>5} ({src}):  '
              f'{["S"+str(i) for i in rank]}')
    comp_rank = np.argsort(comp_per_stab[:, -1])[::-1]
    print(f'  beta=20 (comp):       {["S"+str(i) for i in comp_rank]}')

In [ ]:
"""Publication figure: four-panel summary of the alpha-beta duality."""

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
stab_labels = [f'$S_{i}$ (complement $q_{COMPLEMENT_QUBITS[i]}$)'
               for i in range(N_STAB)]

plot_alpha  = ALPHA_VALUES[alpha_indices]
plot_labels = [ALPHA_LABELS[i] for i in alpha_indices]

# (a) Per-stabilizer fan-out vs alpha
ax = axes[0, 0]
for si in range(N_STAB):
    ax.errorbar(plot_alpha, per_stab_dH[si, alpha_indices],
                yerr=per_stab_sem[si, alpha_indices],
                fmt='o-', color=colors[si], lw=2, ms=6, capsize=3,
                label=stab_labels[si])
    if proj_loaded:
        ax.plot(ALPHA_VALUES[5], per_stab_dH[si, 5], 's',
                color=colors[si], ms=10, zorder=5)
ax.set_xlabel(r'Measurement strength $\alpha$', fontsize=12)
ax.set_ylabel(r'Per-stabilizer $\Delta H$ (bits)', fontsize=12)
ax.set_title(r'(a) Per-stabilizer fan-out vs $\alpha$', fontsize=13, fontweight='bold')
ax.set_xticks(plot_alpha); ax.set_xticklabels(plot_labels, fontsize=9)
ax.legend(fontsize=8, loc='lower left')
ax.grid(True, alpha=0.3)

# (b) Spread and fractional anisotropy
ax = axes[0, 1]
ax2 = ax.twinx()
ax.plot(plot_alpha, spread[alpha_indices], 'o-', color='#2171b5',
        lw=2.5, ms=8, label='Spread (bits)')
ax2.plot(plot_alpha, fractional_anisotropy[alpha_indices], 's--',
         color='#cb181d', lw=2, ms=7, label='Fractional anisotropy (%)')
ax.set_xlabel(r'Measurement strength $\alpha$', fontsize=12)
ax.set_ylabel('Spread (bits)', fontsize=12, color='#2171b5')
ax2.set_ylabel('Fractional anisotropy (%)', fontsize=12, color='#cb181d')
ax.set_title('(b) Fan-out metrics', fontsize=13, fontweight='bold')
ax.set_xticks(plot_alpha); ax.set_xticklabels(plot_labels, fontsize=9)
ax.grid(True, alpha=0.3)
lines1, lab1 = ax.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, lab1 + lab2, fontsize=9, loc='upper left')

# (c) Alpha-beta duality scatter
ax = axes[1, 0]
comp_data = np.load(os.path.join(DATA_DIR, 'fanout_diagnostic_results.npz'),
                    allow_pickle=True)
comp_vals = comp_data['happy_means'][:, -1]
phys_vals = per_stab_dH[:, 5] if proj_loaded else per_stab_dH[:, alpha_indices[-1]]
lo = min(comp_vals.min(), phys_vals.min()) - 0.02
hi = max(comp_vals.max(), phys_vals.max()) + 0.02
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=0.4, label='$y = x$')
for si in range(N_STAB):
    ax.scatter(comp_vals[si], phys_vals[si], color=colors[si], s=120,
               zorder=5, edgecolors='black', lw=0.8, label=stab_labels[si])
spread_ratio = ((phys_vals.max()-phys_vals.min())
                / (comp_vals.max()-comp_vals.min()))
ax.text(0.05, 0.92, f'Spread ratio = {spread_ratio:.3f}',
        transform=ax.transAxes, fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.8))
ax.set_xlabel(r'Computational $\beta$ sweep $\Delta H$ (bits)', fontsize=12)
ax.set_ylabel(r'Physical $\alpha$ sweep $\Delta H$ (bits)', fontsize=12)
ax.set_title(r'(c) $\alpha$–$\beta$ duality: per-stabilizer agreement',
             fontsize=13, fontweight='bold')
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

# (d) Complement qubit mapping (rho vs alpha)
ax = axes[1, 1]
valid = ~np.isnan(rho_per_alpha)
ax.plot(ALPHA_VALUES[valid], rho_per_alpha[valid], 'o-',
        color='steelblue', lw=2, ms=8)
ax.axhline(1.0, color='gray', ls=':', alpha=0.5)
ax.axvspan(0, np.pi/4 - 0.05, alpha=0.08, color='red',
           label=r'$P_{\mathrm{detect}} < 50\%$')
ax.axvspan(np.pi/4 - 0.05, np.pi/2 + 0.1, alpha=0.08, color='green',
           label=r'$P_{\mathrm{detect}} \geq 50\%$')
for ai in alpha_indices:
    if not np.isnan(rho_per_alpha[ai]):
        ax.annotate(f'p={p_per_alpha[ai]:.3f}',
                    (ALPHA_VALUES[ai], rho_per_alpha[ai]),
                    textcoords='offset points', xytext=(0, -15),
                    fontsize=8, ha='center')
ax.set_xlabel(r'Measurement strength $\alpha$', fontsize=12)
ax.set_ylabel(r'Spearman $\rho$(syndrome error, $\Delta H$)', fontsize=12)
ax.set_title(r'(d) Complement qubit mapping vs $\alpha$',
             fontsize=13, fontweight='bold')
ax.set_xticks(plot_alpha); ax.set_xticklabels(plot_labels, fontsize=9)
ax.set_ylim(0.5, 1.1)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(DATA_DIR, 'fig3_alpha_beta_duality.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig_path}')

In [ ]:
"""Save processed results."""

npz_path = os.path.join(DATA_DIR, 'weak_measurement_results.npz')
np.savez(npz_path,
    alpha_values=ALPHA_VALUES,
    alpha_indices=np.array(alpha_indices),
    global_dH=global_dH,
    global_sem=global_sem,
    per_stab_dH=per_stab_dH,
    per_stab_sem=per_stab_sem,
    spread=spread,
    fractional_anisotropy=fractional_anisotropy,
    rho_per_alpha=rho_per_alpha,
    p_per_alpha=p_per_alpha,
    proj_loaded=np.array(proj_loaded),
)
print(f'Results saved: {npz_path}')

# Summary
print(f'\n--- Summary ---')
print(f'Fan-out spread increases monotonically: '
      f'{spread[0]:.4f} (pi/16) -> {spread[alpha_indices[-1]]:.4f} (projective)')
if comp_available:
    print(f'Physical/computational spread ratio at projective limit: '
          f'{spread[alpha_indices[-1]]/comp_spread_sat:.3f}')
print(f'Complement qubit mapping: rho=1.000 at all alpha >= pi/4 '
      f'(Fisher combined p={p_combined:.4f})')
print(f'Global DeltaH decreases with alpha (dual of beta sweep): '
      f'{global_dH[0]:.3f} -> {global_dH[alpha_indices[-1]]:.3f} bits')